In [1]:
import pandas as pd
from gurobipy import Model, GRB, quicksum
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import geopandas as gpd
import plotly.express as px
from unidecode import unidecode

In [2]:
# ------------------ DADOS DE ENTRADA ------------------ #

# Conjunto de distritos
I = list(range(1, 97))
J = I

# Capacidades dos Núcleos
Cap = {j: 2300 for j in J}

# Matriz de distâncias dos distritos
matriz_df = pd.read_csv("matriz_distancias_distritos.csv", index_col=0)

# Padronizar nomes da matriz (maiúsculas, sem acento)
matriz_df.index = (
    matriz_df.index.str.upper()
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("ascii")
)

# Lista de distritos na ordem da matriz
distritos_sp = list(matriz_df.index)

# Ler CSV da Demanda PopRua
demanda_df = pd.read_csv("demanda_por_distrito.csv", sep=";")

# Padronizar nomes dos distritos no CSV da Demanda PopRua
demanda_df["distrito"] = (
    demanda_df["distrito"].str.upper()
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("ascii")
)

demanda_df = demanda_df.set_index("distrito")

# Padronização dos nomes do distrito São Miguel Paulista -> São Miguel
demanda_df = demanda_df.rename(index={"SAO MIGUEL PAULISTA": "SAO MIGUEL"})

# Aferição dos distritos sobre a necessidade da demanda
missing = set(distritos_sp) - set(demanda_df.index)
if missing:
    raise ValueError(f"Distritos sem demanda no CSV: {missing}")

# Reordenamento da demanda na mesma ordem de distritos_sp
demanda_df = demanda_df.loc[distritos_sp]

# Demanda D[i]
D = {i: float(demanda_df.iloc[i-1]["demanda"]) for i in I}

# Matriz de distâncias dist_ij
dist = {i: {} for i in I}
for i_idx, i in enumerate(I):
    distrito_i = distritos_sp[i_idx]
    for j_idx, j in enumerate(J):
        distrito_j = distritos_sp[j_idx]
        dist[i][j] = float(matriz_df.loc[distrito_i, distrito_j])

In [3]:
# ------------------ CRIAR MODELO NO GUROBI ------------------ #

m = Model("Localizacao_14_Nucleos")

# ------------------ VARIÁVEIS DE DECISÃO ------------------ #

# PopRua do distrito i atendido no Núcleo j

x = m.addVars(I, J, vtype=GRB.INTEGER, name="x")

# y[j] = 1 se Núcleo j é aberto
y = m.addVars(J, vtype=GRB.BINARY, name="y")

m.update()

Set parameter Username
Set parameter LicenseID to value 2742323
Academic license - for non-commercial use only - expires 2026-11-20


In [4]:
# ------------------ FUNÇÃO OBJETIVO ------------------ #

# Minimizar a soma das distâncias

m.setObjective(
    quicksum(dist[i][j] * x[i, j] for i in I for j in J),
    GRB.MINIMIZE
)


# ------------------ RESTRIÇÕES ------------------ #

# (1) Atender demanda de cada distrito i
m.addConstrs(
    (quicksum(x[i, j] for j in J) == D[i] for i in I),
    name="demanda_distritos"
)

# (2) Capacidade das vagas dos Núcleos de Convivência

m.addConstrs(
    (quicksum(x[i, j] for i in I) <= Cap[j] * y[j] for j in J),
    name="capacidade_nucleos"
)

# (3) Abrir pelo menos 1 e no máximo 14 Núcleos

m.addConstr(
    quicksum(y[j] for j in J) == 14,
    name="qtd_nucleos_14"
)

# ------------------ OTIMIZAÇÃO ------------------ #

m.optimize()

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i3-8130U CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 193 rows, 9312 columns and 18624 nonzeros (Min)
Model fingerprint: 0x906d6efd
Model has 9120 linear objective coefficients
Variable types: 0 continuous, 9312 integer (96 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [2e+03, 8e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+03]
Found heuristic solution: objective 6.139215e+08
Presolve removed 2 rows and 192 columns
Presolve time: 3.35s
Presolved: 191 rows, 9120 columns, 18240 nonzeros
Variable types: 0 continuous, 9120 integer (192 binary)

Root relaxation: objective 6.765054e+06, 196 iterations, 0.09 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl | 

     0     0 6.0777e+07    0  176 1.3464e+08 6.0777e+07  54.9%     -    5s
H    0     0                    1.341439e+08 6.0929e+07  54.6%     -    5s
H    0     0                    1.160837e+08 6.0929e+07  47.5%     -    5s
H    0     0                    1.160010e+08 6.0929e+07  47.5%     -    5s
H    0     0                    1.159559e+08 6.0929e+07  47.5%     -    5s
     0     0 6.7470e+07    0  225 1.1596e+08 6.7470e+07  41.8%     -    5s
H    0     0                    1.067090e+08 6.7539e+07  36.7%     -    5s
     0     0 6.8048e+07    0  223 1.0671e+08 6.8048e+07  36.2%     -    5s
     0     0 6.8048e+07    0  224 1.0671e+08 6.8048e+07  36.2%     -    5s
     0     0 7.3912e+07    0  274 1.0671e+08 7.3912e+07  30.7%     -    6s
     0     0 7.4132e+07    0  268 1.0671e+08 7.4132e+07  30.5%     -    6s
     0     0 7.4132e+07    0  270 1.0671e+08 7.4132e+07  30.5%     -    6s
     0     0 7.8310e+07    0  312 1.0671e+08 7.8310e+07  26.6%     -    6s
     0     0 7.8551e+07  

In [5]:
import plotly.graph_objects as go
from gurobipy import GRB

if m.Status != GRB.OPTIMAL:
    print(f"Modelo não está ótimo. Status = {m.Status}")
else:
    labels = []
    sources = []
    targets = []
    values = []
    node_index = {}

    # distritos
    for i in I:
        nome_dist = distritos_sp[i-1]
        node_index[("D", i)] = len(labels)
        labels.append(f"Distrito {nome_dist}")

    # todos os núcleos 1..14
    for j in J:
        node_index[("N", j)] = len(labels)
        labels.append(f"Núcleo {j}")

    # fluxos positivos
    for i in I:
        for j in J:
            flow = x[i, j].X
            if flow > 1e-6:
                sources.append(node_index[("D", i)])
                targets.append(node_index[("N", j)])
                values.append(flow)

    print("Qtde de links no Sankey:", len(values))
    if len(values) == 0:
        print("Nenhum fluxo x[i,j] > 0 na solução.")
    else:
        fig = go.Figure(data=[go.Sankey(
            node=dict(
                pad=20,
                thickness=25,
                line=dict(color="black", width=0.5),
                label=labels
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
                color="rgba(0, 100, 200, 0.5)"
            )
        )])

        fig.update_layout(
            title_text=" Abertura para 14 Núcleos de Convivência",
            font_size=12
        )
        fig.write_html("sankey_localizacao.html", auto_open=True)


Qtde de links no Sankey: 105


In [10]:
# ------------------ MAPA DE DEMANDA EM FAIXAS + NÚCLEOS REAIS + NÚCLEOS OTIMIZADOS ------------------

# 1) Ler GeoJSON dos distritos de SP
gdf_dist = gpd.read_file("geoportal_distrito_municipal_v2.geojson")
col_geo_nome = "nm_distrito_municipal"

# 2) Normalizar nomes do GeoJSON
gdf_dist["nome_norm"] = gdf_dist[col_geo_nome].apply(
    lambda x: unidecode(str(x)).upper().strip()
)

# 3) Usar a demanda_df já criada no notebook
demanda_por_distrito = (
    demanda_df["demanda"]
    .astype(float)
    .rename("demanda_total")
    .reset_index()
    .rename(columns={"distrito": "nome_norm"})
)

# 4) Juntar demanda ao GeoDataFrame
gdf_dist = gdf_dist.merge(demanda_por_distrito, on="nome_norm", how="left")

# 5) Criar faixas de demanda
def classificar_demanda(x):
    if pd.isna(x):
        return "Sem informação"
    elif x == 0:
        return "Sem demanda"
    elif x <= 205:
        return "1 - 205"
    elif x <= 590:
        return "206 - 590"
    elif x <= 1255:
        return "591 - 1255"
    elif x <= 2650:
        return "1256 - 2650"
    else:
        return "2651 - 5000"

gdf_dist["faixa_demanda"] = gdf_dist["demanda_total"].apply(classificar_demanda)

# 6) Lista dos 14 distritos com núcleo otimizado
distritos_com_nucleo = [
    "ITAQUERA",
    "REPUBLICA",
    "VILA FORMOSA",
    "LAPA",
    "SANTANA",
    "MOOCA",
    "BOM RETIRO",
    "SANTO AMARO",
    "VILA MARIANA",
    "SANTA CECILIA",
    "CONSOLACAO",
    "BRAS",
    "PARI",
    "SE",
]

distritos_norm = [unidecode(n).upper().strip() for n in distritos_com_nucleo]

gdf_dist["tem_nucleo"] = gdf_dist["nome_norm"].apply(
    lambda n: 1 if n in distritos_norm else 0
)

# 7) Reprojetar se necessário
if gdf_dist.crs and gdf_dist.crs.to_epsg() != 4326:
    gdf_dist = gdf_dist.to_crs(epsg=4326)

# 8) Calcular centróides para os pins
gdf_dist["centroid"] = gdf_dist.geometry.centroid
gdf_dist["lon"] = gdf_dist["centroid"].x
gdf_dist["lat"] = gdf_dist["centroid"].y

# 9) Núcleos otimizados
gdf_pins_otimizados = gdf_dist[gdf_dist["tem_nucleo"] == 1].copy()

# 10) Lista dos núcleos nos locais reais
nucleos_reais = [
    {"id_real": 88, "distrito": "BRAS",         "nome": "Casa Restaura-Me"},
    {"id_real": 49, "distrito": "BELEM",        "nome": "Centro Comunitário São Martinho de Lima"},
    {"id_real": 31, "distrito": "GUAIANASES",   "nome": "Núcleo de Convivência São Rafael"},
    {"id_real": 54, "distrito": "SAO MATEUS",   "nome": "Núcleo Pop São Mateus – Pipas"},
    {"id_real": 60, "distrito": "BELA VISTA",   "nome": "Núcleo Inforedes – Bela Vista"},
    {"id_real": 60, "distrito": "BELA VISTA",   "nome": "Núcleo de Convivência Dom Orione"},
    {"id_real": 72, "distrito": "BOM RETIRO",   "nome": "Casa de Convivência Porto Seguro"},
    {"id_real": 72, "distrito": "BOM RETIRO",   "nome": "Núcleo Prates"},
    {"id_real": 77, "distrito": "BARRA FUNDA",  "nome": "Núcleo Boracea"},
    {"id_real": 94, "distrito": "SE",           "nome": "Núcleo Chá do Padre"},
    {"id_real": 30, "distrito": "REPUBLICA",    "nome": "Núcleo Boticário"},
    {"id_real": 94, "distrito": "SE",           "nome": "Núcleo Rodrigo Silva"},
    {"id_real": 51, "distrito": "SANTANA",      "nome": "Núcleo Santana"},
    {"id_real": 37, "distrito": "PINHEIROS",    "nome": "Núcleo de Convivência Pinheiros"},
]

df_nucleos_reais = pd.DataFrame(nucleos_reais)
df_nucleos_reais["nome_norm"] = df_nucleos_reais["distrito"].apply(
    lambda x: unidecode(str(x)).upper().strip()
)

# 11) Associar cada núcleo real ao centróide do distrito correspondente
gdf_pins_reais_base = gdf_dist[["nome_norm", col_geo_nome, "lon", "lat"]].copy()

gdf_pins_reais = df_nucleos_reais.merge(
    gdf_pins_reais_base,
    on="nome_norm",
    how="left"
)

# 12) Deslocar ligeiramente pontos repetidos no mesmo distrito
gdf_pins_reais["ordem_no_distrito"] = gdf_pins_reais.groupby("nome_norm").cumcount()
gdf_pins_reais["qtd_no_distrito"] = gdf_pins_reais.groupby("nome_norm")["nome_norm"].transform("count")

def deslocamento(i, n, passo=0.01):
    if n == 1:
        return 0
    return (i - (n - 1) / 2) * passo

gdf_pins_reais["lon_plot"] = gdf_pins_reais.apply(
    lambda row: row["lon"] + deslocamento(row["ordem_no_distrito"], row["qtd_no_distrito"], passo=0.015),
    axis=1
)

gdf_pins_reais["lat_plot"] = gdf_pins_reais.apply(
    lambda row: row["lat"] + deslocamento(row["ordem_no_distrito"], row["qtd_no_distrito"], passo=0.008),
    axis=1
)

# 13) Paleta da legenda
cores_faixas = {
    "Sem demanda": "#FFFFFF",
    "1 - 205": "#F9E9D2",
    "206 - 590": "#F1C9A7",
    "591 - 1255": "#E9A177",
    "1256 - 2650": "#D96543",
    "2651 - 5000": "#A61E22",
    "Sem informação": "#F2F2F2"
}

ordem_faixas = [
    "Sem demanda",
    "1 - 205",
    "206 - 590",
    "591 - 1255",
    "1256 - 2650",
    "2651 - 5000",
    "Sem informação"
]

# 14) Criar mapa categórico
fig = px.choropleth(
    gdf_dist,
    geojson=gdf_dist.geometry.__geo_interface__,
    locations=gdf_dist.index,
    color="faixa_demanda",
    category_orders={"faixa_demanda": ordem_faixas},
    color_discrete_map=cores_faixas,
    hover_name=col_geo_nome,
    hover_data={
        "demanda_total": True,
        "faixa_demanda": True,
        "tem_nucleo": True,
        "nome_norm": False
    }
)

fig.update_geos(fitbounds="locations", visible=False)

# 15) Trace para legenda dos núcleos otimizados
fig.add_trace(
    go.Scattergeo(
        lon=[None],
        lat=[None],
        mode="markers",
        marker=dict(
            size=10,
            color="green",
            symbol="diamond",
            line=dict(width=1, color="white"),
        ),
        name="Núcleo otimizado",
        showlegend=True
    )
)

# 16) Trace real dos núcleos otimizados
fig.add_trace(
    go.Scattergeo(
        lon=gdf_pins_otimizados["lon"],
        lat=gdf_pins_otimizados["lat"],
        mode="markers",
        marker=dict(
            size=10,
            color="green",
            symbol="diamond",
            line=dict(width=1, color="white"),
        ),
        showlegend=False,
        hovertext=gdf_pins_otimizados[col_geo_nome],
        hoverinfo="text"
    )
)

# 17) Trace para legenda dos núcleos reais
fig.add_trace(
    go.Scattergeo(
        lon=[None],
        lat=[None],
        mode="markers",
        marker=dict(
            size=10,
            color="red",
            symbol="circle",
            line=dict(width=1, color="white"),
        ),
        name="Núcleo real",
        showlegend=True
    )
)

# 18) Trace real dos núcleos reais
fig.add_trace(
    go.Scattergeo(
        lon=gdf_pins_reais["lon_plot"],
        lat=gdf_pins_reais["lat_plot"],
        mode="markers",
        marker=dict(
            size=10,
            color="red",
            symbol="circle",
            line=dict(width=1, color="white"),
        ),
        showlegend=False,
        text=gdf_pins_reais["nome"],
        customdata=gdf_pins_reais[["distrito", "id_real"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Distrito: %{customdata[0]}<br>"
            "ID real: %{customdata[1]}<extra></extra>"
        )
    )
)

# 19) Layout final
fig.update_layout(
    title="Distribuição da população em situação de rua por distrito e localização dos núcleos reais e otimizados",
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    legend_title_text="Legenda"
)

# 20) Salvar
fig.write_html("mapa_demanda_faixas_nucleos_reais_e_otimizados.html", auto_open=True)

C:\Users\vmlle\AppData\Local\Temp\ipykernel_4420\2112089021.py:72: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


